# Version 0.5

# LLM Benchmark on Google Colab T4 GPU

This notebook demonstrates how to:
1. Load a pre-trained LLM model
2. Run inference benchmarks on Google Colab's T4 GPU
3. Measure performance metrics (latency, throughput, memory usage)
4. Visualize the results

The model used is **Mistral 7B Instruct v0.3** - a public instruction-tuned Mistral checkpoint that is loaded in 4-bit so it can run on a Colab T4 GPU.

## 1. Install Required Libraries

In [ ]:
# Install required libraries
import subprocess
import sys

# Install transformers and torch
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "accelerate"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "datasets"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "matplotlib"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-learn"])

print("✓ All libraries installed successfully!")

## 2. Check GPU and Setup

In [ ]:
import os
import torch
import numpy as np
import time
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime

# Check GPU availability
print("GPU Information:")
print(f"  Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"  CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"  Current GPU Memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print()

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Using device: {device}")

## 3. Load Pretrained LLM Model

We use **Mistral 7B Instruct v0.3** in a quantized configuration. That keeps the notebook usable on a T4 while switching from classifier-style inference to prompt-based multiple-choice scoring.

In [ ]:
# Load Mistral 7B Instruct v0.3 tokenizer and model
print("Loading Mistral 7B Instruct v0.3 model...")
model_name = "mistralai/Mistral-7B-Instruct-v0.3"
hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")

tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

use_quantization = torch.cuda.is_available()
if use_quantization:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quantization_config,
        device_map="auto",
        token=hf_token,
    )
else:
    model = AutoModelForCausalLM.from_pretrained(model_name, token=hf_token)
    model = model.to(device)

model.eval()  # Set to evaluation mode

total_parameters = sum(p.numel() for p in model.parameters())
param_size = total_parameters * (0.5 if use_quantization else 4) / (1024 ** 2)

print(f"✓ Model loaded successfully!")
print(f"  Model: {model_name}")
print(f"  Total parameters: {total_parameters / 1e9:.2f}B")
print(f"  Approx. model size: {param_size:.2f} MB")

## 4. Load ARC (AI2 Reasoning Challenge) Dataset

The ARC dataset contains science exam questions with 4 multiple-choice answers. It tests reasoning and knowledge across science, physics, chemistry, and biology.

In [ ]:
from datasets import load_dataset

# Load ARC dataset (Easy and Challenge)
print("Loading ARC dataset...")
arc_easy = load_dataset("ai2_arc", "ARC-Easy", split="test", trust_remote_code=True)
arc_challenge = load_dataset("ai2_arc", "ARC-Challenge", split="test", trust_remote_code=True)

# Combine and limit to 600 samples each for T4 GPU (total 1200 questions)
arc_data = []

# Use .select() to properly index dataset
easy_indices = range(min(600, len(arc_easy)))
challenge_indices = range(min(600, len(arc_challenge)))

for idx in easy_indices:
    sample = dict(arc_easy[idx])
    sample["source"] = "ARC-Easy"
    arc_data.append(sample)

for idx in challenge_indices:
    sample = dict(arc_challenge[idx])
    sample["source"] = "ARC-Challenge"
    arc_data.append(sample)

def normalize_arc_sample(sample, sample_index):
    question_text = sample["question"]
    if isinstance(question_text, dict):
        question_text = question_text.get("stem", str(question_text))

    choices = sample["choices"]
    if isinstance(choices, dict):
        choice_labels = choices.get("label", [])
        choice_texts = choices.get("text", [])
        normalized_choices = [
            {"label": label, "text": text}
            for label, text in zip(choice_labels, choice_texts)
        ]
    else:
        normalized_choices = [
            {"label": choice["label"], "text": choice["text"]}
            for choice in choices
        ]

    return {
        "question_id": sample.get("id", f"question_{sample_index}"),
        "category": sample.get("source", "ARC"),
        "question": question_text,
        "choices": normalized_choices,
        "answerKey": sample["answerKey"],
    }

arc_questions = [normalize_arc_sample(sample, idx) for idx, sample in enumerate(arc_data)]

print(f"✓ Loaded {len(arc_questions)} ARC questions (600 Easy + 600 Challenge)")
print(f"\nFirst example:")
print(f"  Question: {arc_questions[0]['question']}")
print(f"  Choices: {[choice['text'] for choice in arc_questions[0]['choices']]}")
print(f"  Answer: {arc_questions[0]['answerKey']}")

## 5. Run ARC Benchmark

Evaluate the model on ARC questions - measure accuracy and reasoning capability.

In [ ]:
# Evaluate model on ARC dataset
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from collections import defaultdict

print("Running ARC Benchmark...\n")
print("=" * 60)

def build_prompt(sample):
    option_lines = "\n".join(
        f"{choice['label']}. {choice['text']}" for choice in sample['choices']
    )
    return (
        "You are solving a multiple-choice science question.\n"
        f"Question: {sample['question']}\n"
        f"Options:\n{option_lines}\n\n"
        "Reply with only the answer letter."
    )

# Flatten questions into question-choice samples for scoring
arc_samples = []
for question_index, sample in enumerate(arc_questions):
    prompt = build_prompt(sample)
    for choice_idx, choice in enumerate(sample['choices']):
        arc_samples.append({
            'question_id': sample['question_id'],
            'category': sample['category'],
            'question': sample['question'],
            'choices': sample['choices'],
            'choice_idx': choice_idx,
            'choice_label': choice['label'],
            'choice': choice['label'],
            'completion': f" {choice['label']}",
            'prompt': prompt,
            'label': 1 if choice['label'] == sample['answerKey'] else 0,
            'answerKey': sample['answerKey'],
        })

question_predictions = defaultdict(list)
category_accuracies = defaultdict(lambda: {'correct': 0, 'total': 0})
timings = []
all_predictions = []
all_labels = []

print(f"Processing {len(arc_samples)} question-choice pairs...")
print(f"Batch processing with batch size: 8\n")

batch_size = 8

for batch_idx in range(0, len(arc_samples), batch_size):
    batch_samples = arc_samples[batch_idx:batch_idx + batch_size]
    batch_texts = [sample['prompt'] + sample['completion'] for sample in batch_samples]
    prompt_lengths = [len(tokenizer(sample['prompt']).input_ids) for sample in batch_samples]
    full_lengths = [len(tokenizer(text).input_ids) for text in batch_texts]

    encodings = tokenizer(
        batch_texts,
        truncation=True,
        padding=True,
        max_length=384,
        return_tensors="pt"
    )

    input_ids = encodings["input_ids"].to(device)
    attention_mask = encodings["attention_mask"].to(device)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start_time = time.time()

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        log_probs = torch.log_softmax(logits[:, :-1, :], dim=-1)
        target_ids = input_ids[:, 1:]
        token_log_probs = log_probs.gather(-1, target_ids.unsqueeze(-1)).squeeze(-1)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    end_time = time.time()
    timings.append(end_time - start_time)

    for i, sample in enumerate(batch_samples):
        prompt_length = prompt_lengths[i]
        full_length = full_lengths[i]
        completion_length = max(full_length - prompt_length, 1)
        start_index = max(prompt_length - 1, 0)
        end_index = min(start_index + completion_length, token_log_probs.shape[1])
        score = token_log_probs[i, start_index:end_index].sum().item()

        question_predictions[sample['question_id']].append({
            'choice_idx': sample['choice_idx'],
            'choice_label': sample['choice_label'],
            'confidence': score,
            'label': sample['label'],
            'category': sample['category'],
            'answerKey': sample['answerKey'],
        })

print(f"\n✓ Inference complete!")
print(f"Total inference time: {sum(timings):.2f}s")
print(f"Average time per sample: {(sum(timings) / len(arc_samples) * 1000):.2f}ms")

predictions_per_question = []
labels_per_question = []
per_sample_preds = []
per_sample_labels = []

for q_id, choices in question_predictions.items():
    best_choice = max(choices, key=lambda x: x['confidence'])
    predicted_choice = best_choice['choice_label']
    ground_truth_choice = best_choice['answerKey']
    category = best_choice['category']

    predictions_per_question.append(predicted_choice)
    labels_per_question.append(ground_truth_choice)

    for choice in choices:
        per_sample_preds.append(choice['choice_label'] == predicted_choice)
        per_sample_labels.append(choice['label'])

    category_accuracies[category]['total'] += 1
    if predicted_choice == ground_truth_choice:
        category_accuracies[category]['correct'] += 1

accuracy_per_question = accuracy_score(labels_per_question, predictions_per_question)
accuracy_per_choice = accuracy_score(per_sample_labels, per_sample_preds)

print(f"\nAccuracy Metrics:")
print(f"  Per-Question Accuracy: {accuracy_per_question * 100:.2f}%")
print(f"  Per-Choice Accuracy: {accuracy_per_choice * 100:.2f}%")

category_accuracies = dict(category_accuracies)

benchmark_results = {
    'Accuracy (per question)': accuracy_per_question,
    'Accuracy (per choice)': accuracy_per_choice,
    'Total Questions': len(predictions_per_question),
    'Total Samples': len(arc_samples),
    'Avg Inference Time (ms)': (sum(timings) / len(arc_samples)) * 1000,
    'Total Inference Time (s)': sum(timings),
    'Category Accuracies': category_accuracies
}

print("\n✓ Benchmark complete!")

## 6. Display Results

Show detailed benchmark results including accuracy scores and performance metrics.

In [ ]:
# Display detailed results
print("\n" + "=" * 80)
print("Mistral 7B Instruct v0.3 ARC BENCHMARK RESULTS (120+ Questions)")
print("=" * 80)

print(f"\nDataset Statistics:")
print(f"  Total Questions: {benchmark_results['Total Questions']}")
print(f"  Total Samples: {benchmark_results['Total Samples']}")
print(f"  Question-Answer Pairs: {len(arc_questions)}")

print(f"\nOverall Accuracy Scores:")
print(f"  Per-Question Accuracy: {benchmark_results['Accuracy (per question)'] * 100:.2f}%")
print(f"  Per-Choice Accuracy: {benchmark_results['Accuracy (per choice)'] * 100:.2f}%")

print(f"\nAccuracy by Category:")
print("-" * 80)
print(f"{'Category':<30} {'Accuracy':<15} {'Correct/Total':<20}")
print("-" * 80)

for category, stats in sorted(benchmark_results['Category Accuracies'].items(),
                              key=lambda x: (x[1]['total']/x[1]['total'] if x[1]['total'] > 0 else 0) *
                                           (x[1]['correct']/x[1]['total'] if x[1]['total'] > 0 else 0),
                              reverse=True):
    if stats['total'] > 0:
        cat_acc = (stats['correct'] / stats['total']) * 100
        print(f"{category:<30} {cat_acc:>6.2f}%{'':<7} {stats['correct']:>2}/{stats['total']:<10}")

print("-" * 80)

print(f"\nPerformance Metrics:")
print(f"  Avg Inference Time: {benchmark_results['Avg Inference Time (ms)']:.2f} ms")
print(f"  Total Inference Time: {benchmark_results['Total Inference Time (s)']:.2f} s")
print(f"  Samples per Second: {benchmark_results['Total Samples'] / benchmark_results['Total Inference Time (s)']:.2f}")

if torch.cuda.is_available():
    peak_memory = torch.cuda.max_memory_allocated() / 1024 ** 2
    current_memory = torch.cuda.memory_allocated() / 1024 ** 2
    total_memory = torch.cuda.get_device_properties(0).total_memory / 1024 ** 2

    print(f"\nGPU Memory Usage:")
    print(f"  Peak Memory: {peak_memory:.2f} MB")
    print(f"  Current Memory: {current_memory:.2f} MB")
    print(f"  Total GPU Memory: {total_memory:.2f} MB")
    print(f"  Memory Utilization: {(peak_memory / total_memory) * 100:.2f}%")

print(f"\nModel Information:")
print(f"  Model: {model_name}")
print(f"  Total Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")
print(f"  Approx. Model Size: {param_size:.2f} MB")

print("\n" + "=" * 80)

## 7. Visualize Results

Create plots to display benchmark results for easy interpretation.

In [ ]:
# Create detailed visualizations for ARC benchmark results with individual category plots
# Prepare category data
categories_list = []
accuracies_list = []
correct_counts = []
total_counts = []

for category, stats in sorted(benchmark_results['Category Accuracies'].items()):
    if stats['total'] > 0:
        categories_list.append(category)
        acc = (stats['correct'] / stats['total']) * 100
        accuracies_list.append(acc)
        correct_counts.append(stats['correct'])
        total_counts.append(stats['total'])

num_categories = len(categories_list)
num_cols = 3
num_rows = (num_categories + num_cols - 1) // num_cols + 2  # +2 for overall plots

fig = plt.figure(figsize=(18, 4 * num_rows))
gs = fig.add_gridspec(num_rows, num_cols, hspace=0.4, wspace=0.3)

fig.suptitle("DistilBERT Performance on ARC - Detailed Category Analysis (120+ Questions)",
             fontsize=18, fontweight="bold", y=0.995)

# Plot 1: Overall Accuracy (top-left)
ax_overall = fig.add_subplot(gs[0, 0])
overall_accs = [benchmark_results['Accuracy (per question)'] * 100,
                benchmark_results['Accuracy (per choice)'] * 100]
colors_overall = ['steelblue', 'coral']
bars_overall = ax_overall.bar(['Per-Question\nAccuracy', 'Per-Choice\nAccuracy'], overall_accs,
                              color=colors_overall, alpha=0.7, edgecolor='black', linewidth=1.5)
ax_overall.set_ylabel('Accuracy (%)', fontsize=11, fontweight="bold")
ax_overall.set_title('Overall ARC Accuracy', fontsize=12, fontweight="bold")
ax_overall.set_ylim([0, 100])
ax_overall.grid(True, alpha=0.3, axis='y')
for bar, acc in zip(bars_overall, overall_accs):
    height = bar.get_height()
    ax_overall.text(bar.get_x() + bar.get_width()/2., height, f'{acc:.1f}%',
                   ha='center', va='bottom', fontsize=10, fontweight='bold')

# Plot 2: Category Accuracies Overview (top-middle and top-right)
ax_overview = fig.add_subplot(gs[0, 1:])
if categories_list:
    colors_cat = plt.cm.RdYlGn(np.array(accuracies_list) / 100.0)
    bars_overview = ax_overview.barh(range(len(categories_list)), accuracies_list,
                                     color=colors_cat, alpha=0.8, edgecolor='black', linewidth=1.5)
    ax_overview.set_xlabel('Accuracy (%)', fontsize=11, fontweight="bold")
    ax_overview.set_title('Accuracy Rankings by Category', fontsize=12, fontweight="bold")
    ax_overview.set_yticks(range(len(categories_list)))
    ax_overview.set_yticklabels(categories_list)
    ax_overview.set_xlim([0, 100])
    ax_overview.grid(True, alpha=0.3, axis='x')

    # Add value labels
    for i, (bar, acc, correct, total) in enumerate(zip(bars_overview, accuracies_list, correct_counts, total_counts)):
        width = bar.get_width()
        ax_overview.text(width, bar.get_y() + bar.get_height()/2.,
                        f' {acc:.1f}% ({correct}/{total})',
                        ha='left', va='center', fontsize=9, fontweight='bold')
else:
    ax_overview.text(0.5, 0.5, 'No category data available',
                    ha='center', va='center', transform=ax_overview.transAxes, fontsize=12)
    ax_overview.axis('off')

# Plot individual category plots
for idx, (category, accuracy, correct, total) in enumerate(zip(categories_list, accuracies_list, correct_counts, total_counts)):
    row = 1 + (idx // num_cols)
    col = idx % num_cols
    ax = fig.add_subplot(gs[row, col])

    # Create detailed data for this category
    correct_pct = accuracy
    incorrect_pct = 100 - accuracy

    # Pie chart for correct/incorrect
    sizes = [correct, total - correct]
    labels = [f'Correct\n{correct}', f'Incorrect\n{total - correct}']
    colors_pie = ['#2ecc71', '#e74c3c']
    explode = (0.05, 0.05)

    wedges, texts, autotexts = ax.pie(sizes, explode=explode, labels=labels, colors=colors_pie,
                                        autopct='%1.1f%%', shadow=True, startangle=90,
                                        textprops={'fontsize': 10, 'weight': 'bold'})

    # Make percentage text more visible
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontsize(11)
        autotext.set_weight('bold')

    # Title with category and stats
    ax.set_title(f'{category}\n{correct_pct:.1f}% Accuracy (n={total})',
                fontsize=11, fontweight="bold", pad=10)

plt.show()

print("\n✓ Detailed Visualization complete!")

## About This Benchmark

### What is ARC?
The **AI2 Reasoning Challenge** (ARC) is a benchmark containing science exam questions from 3rd to 9th grade. Questions span:
- **Physics**: Motion, Forces, Energy
- **Chemistry**: Elements, Reactions, Matter
- **Biology**: Life Processes, Genetics, Ecosystems
- **Earth Science**: Weather, Geology, Astronomy
- **General Science**: Problem-Solving, Analysis

### Dataset Details
- **ARC-Easy**: 2,590 questions (easier difficulty)
- **ARC-Challenge**: 1,119 questions (harder difficulty)
- **Format**: Multiple choice with 4 answer options (A, B, C, D)
- **This Benchmark**: Uses 15 Easy + 15 Challenge questions for quick testing

### How This Benchmark Works
1. Each question is rendered as a multiple-choice prompt
2. Mistral scores the answer-letter completions for each option
3. The highest-scoring choice is selected as the answer
4. Accuracy is calculated against ground truth labels

### Note on Mistral 7B Instruct v0.3
- Mistral 7B Instruct v0.3 is loaded in **4-bit** to fit on a Colab T4
- Access may require you to accept the model license on Hugging Face and set `HF_TOKEN` or `HUGGINGFACE_TOKEN`
- It is evaluated with **prompt-based multiple-choice scoring** instead of classifier heads
- This benchmark shows how well the model handles science QA and ARC-style reasoning
- For better results, consider prompt tuning or further fine-tuning on QA data

### How to Use in Google Colab
1. Open [Google Colab](https://colab.research.google.com)
2. Go to **Runtime → Change runtime type → GPU (T4)**
3. Upload this notebook
4. Run all cells: **Runtime → Run all**
5. View accuracy results and performance metrics

### Customization
To use more ARC questions, modify line in "Load ARC Dataset" cell:
```python
arc_easy = load_dataset("ai2_arc", "ARC-Easy", split="test", trust_remote_code=True)[:N]  # Change N
arc_challenge = load_dataset("ai2_arc", "ARC-Challenge", split="test", trust_remote_code=True)[:N]  # Change N
```

### Tips for Better Results
- **Fine-tuning**: Train the model on ARC questions for better accuracy
- **Larger models**: Try a bigger Gemma checkpoint if your GPU allows it
- **Ensemble**: Combine multiple models for better predictions
- **Prompt engineering**: Reformulate questions to improve understanding